# Notebook 1 — `1_preprocessing_bnpb`

## Overview

**Purpose:** Compile and clean annual BNPB flood records for 2016–2025, then match the records with BPS regency and municipality codes.

## Data requirements

Download the following data before running the notebook. The `data/raw/` directory is empty in this repository because raw data are not redistributed in accordance with the providers' terms of use.

| Dataset | Source | Required files | Directory |
|---|---|---|---|
| Annual flood disaster records | [BNPB GIS](https://gis.bnpb.go.id/) | One file per year, 2016–2025 | `data/raw/BNPB/` |
| BPS master codes (Bridging Kode) | [BPS Bridging Kode](https://sig.bps.go.id/bridging-kode/index) | One file per province (38 files) | `data/raw/BPS/BPS_Master_Code/` |

## Workflow

| Step | Description |
|---|---|
| 1 | Import libraries, configure settings, and define helper functions |
| 2 | Define name-correction dictionaries |
| 3 | Read and compile annual Excel files |
| 3b | Audit and explain Kode Identitas Bencana |
| 4 | Perform basic cleaning, including column renaming, date parsing, and string processing |
| 5 | Fill missing values (numeric fields with `0` and text fields with `-`) |
| 6 | Identify and remove duplicate rows |
| 7 | Normalise regency and province names |
| 8 | Build the master regency table |
| 9 | Load BPS master codes and build lookup tables |
| 9b | Dynamically remap Papua codes using the BPS master data |
| 10 | Generate `bnpb_code_new` using three-tier matching |
| 11 | Perform a full outer join and generate comparison and quality-control reports |
| 12 | Add BPS codes to the compiled dataset and export the results |

## Output

`data/processed/1_bencana_compiled_with_bps.xlsx`

## Important note on Kode Identitas Bencana

The Kode Identitas Bencana is a 16-digit reference code. Python reads the actual value stored in the Excel cell, whereas Excel displays only 15 significant digits. Consequently, the final digit may appear as `0` in Excel even when the stored value ends in `1`. For example, the stored value `3315101201612301` may be displayed as `3315101201612300`.

This is an Excel display limitation rather than a pandas issue. The code is used for reference purposes only and is **not used as a join key**.

## STEP 1 — Imports, config, helper functions

In [24]:
# =============================================================================
# STEP 1 -- IMPORTS, CONFIG, HELPER FUNCTIONS
# =============================================================================
import os
import re
import pandas as pd
import numpy as np
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# -- Paths ------------------------------------------------------------------
DATA_DIR    = "../data/raw/BNPB/"
BPS_PATH    = "../data/raw/BPS/BPS_Master_Code/master_code_full.xlsx"
OUTPUT_PATH = "../data/processed/1_bencana_compiled_with_bps.xlsx"

# -- Constants --------------------------------------------------------------
YEARS      = range(2016, 2026)
HEADER_ROW = 1        # 0-indexed -> row 2 in Excel holds column names

# Numeric damage/casualty columns -> fill blank with 0
NUMERIC_COLS = [
    "Meninggal", "Hilang", "Terluka",
    "Rumah Rusak", "Rumah Terendam", "Fasum Rusak",
]

# Free-text narrative columns -> fill blank with "-"
TEXT_COLS = [
    "Lokasi", "Penyebab", "Kronologi & Dokumentasi",
]

# All values considered "blank" in raw string form
BLANK_VALUES = {"", "nan", "none", "null", "n/a", "na", "#n/a"}


# -- Helper functions -------------------------------------------------------

def is_blank(val) -> bool:
    """True if val is genuinely empty/missing in any form."""
    if val is None:
        return True
    if pd.isna(val):
        return True
    return str(val).strip().lower() in BLANK_VALUES


def strip_prefix(name) -> str:
    """Remove Kabupaten / Kota / Kab. prefix, uppercase, strip."""
    if pd.isna(name) or str(name).strip() == "":
        return np.nan
    return re.sub(
        r"^(kabupaten|kota|kab\.?)\s+", "",
        str(name).strip(), flags=re.IGNORECASE
    ).strip().upper()


def reformat_kemendagri_code(code) -> str:
    """'11.09' -> '1109',  '11.1' -> '1110',  '14.72' -> '1472'"""
    if pd.isna(code):
        return np.nan
    code = str(code).strip()
    if "." in code:
        prov, kab = code.split(".", 1)
        kab = kab.strip()
        kab = kab + "0" if len(kab) == 1 else kab.zfill(2)
        return prov.strip().zfill(2) + kab
    return str(int(float(code))).zfill(4)


def write_excel_sheet(ws, df):
    """Write df to openpyxl worksheet with styled header, frozen pane, auto-width."""
    HEADER_FILL = PatternFill("solid", fgColor="1F4E79")
    HEADER_FONT = Font(color="FFFFFF", bold=True)
    THIN = Border(
        left=Side(style="thin"),  right=Side(style="thin"),
        top=Side(style="thin"),   bottom=Side(style="thin"),
    )
    for c, col in enumerate(df.columns, 1):
        cell = ws.cell(row=1, column=c, value=col)
        cell.fill = HEADER_FILL
        cell.font = HEADER_FONT
        cell.alignment = Alignment(horizontal="center", wrap_text=True)
        cell.border = THIN
    for r, (_, row) in enumerate(df.iterrows(), 2):
        for c, col in enumerate(df.columns, 1):
            val = row[col]
            if pd.isna(val):
                val = ""
            cell = ws.cell(row=r, column=c, value=val)
            cell.border = THIN
    for c in range(1, len(df.columns) + 1):
        vals = [
            str(ws.cell(row=r2, column=c).value or "")
            for r2 in range(1, ws.max_row + 1)
        ]
        ws.column_dimensions[get_column_letter(c)].width = min(
            max((len(v) for v in vals), default=10) + 2, 45
        )
    ws.freeze_panes    = "A2"
    ws.auto_filter.ref = ws.dimensions


## STEP 2 — Name correction dictionaries

Manual corrections for regency and province name variants in the BNPB records (spelling, mergers, renames) so names normalise consistently.

In [27]:
# =============================================================================
# STEP 2 -- NAME CORRECTION DICTIONARIES
# =============================================================================

BNPB_REGENCY_CORRECTIONS = {
    "KUTAI KERTANEGARA"               : "KUTAI KARTANEGARA",
    "MAHAKAM HULU"                    : "MAHAKAM ULU",
    "KEPULAUAN SIAU TAGULANDANG BIARO": "KEP. SIAU TAGULANDANG BIARO",
    "TOLI TOLI"                       : "TOLI-TOLI",
    "LABUHAN BATU"                    : "LABUHANBATU",
    "TOBA SAMOSIR"                    : "TOBA",
    "LABUHAN BATU SELATAN"            : "LABUHANBATU SELATAN",
    "LABUHAN BATU UTARA"              : "LABUHANBATU UTARA",
    "KOTA PEMATANG SIANTAR"           : "KOTA PEMATANGSIANTAR",
    "KOTA TANJUNG BALAI"              : "KOTA TANJUNGBALAI",
    "KOTA PADANG SIDEMPUAN"           : "KOTA PADANGSIDIMPUAN",
    "KOTA GUNUNG SITOLI"              : "KOTA GUNUNGSITOLI",
    "MUSI BANYU ASIN"                 : "MUSI BANYUASIN",
    "MUKO MUKO"                       : "MUKOMUKO",
    "KOTA PANGKALPINANG"              : "KOTA PANGKAL PINANG",
    "KULONPROGO"                      : "KULON PROGO",
    "TOJO UNA-UNA"                    : "TOJO UNA UNA",
    "SELAYAR"                         : "KEPULAUAN SELAYAR",
    "PANGKAJENE KEPULAUAN"            : "PANGKAJENE DAN KEPULAUAN",
    "KOTA PARE-PARE"                  : "PAREPARE",
    "KOTA PAREPARE"                   : "PAREPARE",
    "MALUKU TENGGARA BARAT"           : "KEPULAUAN TANIMBAR",
    "KOTA BAUBAU"                     : "BAU BAU",
    "FAK-FAK"                         : "FAK FAK",
    "KOTA ADM. JAKARTA PUSAT"         : "ADMINISTRASI JAKARTA PUSAT",
    "KOTA JAKARTA PUSAT"              : "ADMINISTRASI JAKARTA PUSAT",
    "KOTA ADM. JAKARTA UTARA"         : "ADMINISTRASI JAKARTA UTARA",
    "KOTA JAKARTA UTARA"              : "ADMINISTRASI JAKARTA UTARA",
    "KOTA ADM. JAKARTA BARAT"         : "ADMINISTRASI JAKARTA BARAT",
    "KOTA JAKARTA BARAT"              : "ADMINISTRASI JAKARTA BARAT",
    "KOTA ADM. JAKARTA SELATAN"       : "ADMINISTRASI JAKARTA SELATAN",
    "KOTA JAKARTA SELATAN"            : "ADMINISTRASI JAKARTA SELATAN",
    "KOTA ADM. JAKARTA TIMUR"         : "ADMINISTRASI JAKARTA TIMUR",
    "KOTA JAKARTA TIMUR"              : "ADMINISTRASI JAKARTA TIMUR",
}

BNPB_PROVINCE_CORRECTIONS = {
    "DAERAH ISTIMEWA YOGYAKARTA": "DI YOGYAKARTA",
    "P A P U A"                 : "PAPUA",
}

# Old Papua BNPB code -> regency name (used to build remap dynamically)
PAPUA_OLD_CODE_TO_NAME = {
    "9101": "MERAUKE",           "9102": "JAYAWIJAYA",
    "9103": "JAYAPURA",          "9104": "NABIRE",
    "9105": "KEPULAUAN YAPEN",   "9106": "BIAK NUMFOR",
    "9108": "PANIAI",            "9109": "MIMIKA",
    "9110": "SARMI",             "9111": "KEEROM",
    "9112": "PEGUNUNGAN BINTANG","9113": "YAHUKIMO",
    "9114": "TOLIKARA",          "9115": "WAROPEN",
    "9119": "SUPIORI",           "9120": "MAMBERAMO RAYA",
    "9123": "LANNY JAYA",        "9126": "DOGIYAI",
    "9128": "DEIYAI",
    "9202": "MANOKWARI",         "9203": "FAK FAK",
    "9205": "RAJA AMPAT",        "9206": "TELUK BINTUNI",
    "9207": "TELUK WONDAMA",     "9208": "KAIMANA",
    "9209": "TAMBRAUW",          "9210": "MAYBRAT",
    "9211": "MANOKWARI SELATAN", "9212": "PEGUNUNGAN ARFAK",
    "9301": "MERAUKE",    "9302": "BOVEN DIGOEL", "9303": "MAPPI",    "9304": "ASMAT",
    "9401": "NABIRE",     "9402": "PUNCAK JAYA",  "9403": "PANIAI",   "9404": "MIMIKA",
    "9405": "PUNCAK",     "9406": "DOGIYAI",      "9407": "INTAN JAYA","9408": "DEIYAI",
    "9501": "JAYAWIJAYA", "9502": "PEGUNUNGAN BINTANG","9503": "YAHUKIMO","9504": "TOLIKARA",
    "9505": "MAMBERAMO TENGAH","9506": "YALIMO",   "9507": "LANNY JAYA","9508": "NDUGA",
    "9601": "SORONG",     "9602": "SORONG SELATAN","9603": "RAJA AMPAT",
    "9604": "TAMBRAUW",   "9605": "MAYBRAT",
}

PAPUA_CODE_DIRECT = {
    "9171": "9171",
    "9201": "9601",
    "9271": "9671",
    "9601": "9601",
    "9671": "9671",
}


## STEP 3 — Read and compile annual files

Locate and read each annual flood `.xlsx` file, concatenating them into a single compiled table. All columns are read as strings to preserve raw values before cleaning.

*STEP 3b — Kode Identitas Bencana audit:* see the note at the foot of this notebook. The 16-digit disaster code is a reference ID only and is never used as a join key.

In [30]:
# =============================================================================
# STEP 3 -- READ & COMPILE ANNUAL FILES
# =============================================================================

def find_file(year: int) -> str | None:
    xlsx_files = [f for f in os.listdir(DATA_DIR) if f.lower().endswith(".xlsx")]
    if year == list(YEARS)[0]:
        print(f"  Files in DATA_DIR ({len(xlsx_files)} xlsx):")
        for f in sorted(xlsx_files):
            print(f"    {f}")
    p1 = re.compile(rf"data[\s_-]*bencana[\s_-]*{year}\.xlsx", re.IGNORECASE)
    p2 = re.compile(rf"bencana.*{year}\.xlsx",                  re.IGNORECASE)
    p3 = re.compile(rf".*{year}\.xlsx",                         re.IGNORECASE)
    for pattern in [p1, p2, p3]:
        for name in sorted(xlsx_files):
            if pattern.match(name):
                return os.path.join(DATA_DIR, name)
    return None


def read_year(year: int) -> pd.DataFrame | None:
    path = find_file(year)
    if path is None:
        print(f"  [SKIP] {year}: no xlsx found")
        return None
    try:
        df = pd.read_excel(path, header=HEADER_ROW, dtype=str)
    except Exception as exc:
        print(f"  [ERROR] {year}: {exc}")
        return None
    df.dropna(how="all", inplace=True)
    df.dropna(axis=1, how="all", inplace=True)
    df.columns = df.columns.str.strip()
    df.insert(0, "year", year)
    print(f"  [OK] {year}: {len(df):,} rows  | {os.path.basename(path)}")
    return df


print("=" * 60)
print("STEP 3 -- Read & compile annual files")
print("=" * 60)

frames, missing_years = [], []
for yr in YEARS:
    frame = read_year(yr)
    if frame is not None:
        frames.append(frame)
    else:
        missing_years.append(yr)

if not frames:
    raise SystemExit("No data files found. Check DATA_DIR.")

combined = pd.concat(frames, ignore_index=True)
print(f"\nCombined : {combined.shape[0]:,} rows x {combined.shape[1]} cols")
if missing_years:
    print(f"Missing years: {missing_years}")


# =============================================================================
# STEP 3b -- KODE IDENTITAS BENCANA AUDIT
# =============================================================================

print("\n" + "=" * 60)
print("STEP 3b -- Kode Identitas Bencana audit")
print("=" * 60)

kode_raw = combined["Kode Identitas Bencana"].dropna().astype(str)
wrong_len    = kode_raw[kode_raw.str.len() != 16]
last_nonzero = kode_raw[(kode_raw.str.len() == 16) & (kode_raw.str[-1] != "0")]

print(f"dtype              : {combined['Kode Identitas Bencana'].dtype}  (must be object/string)")
print(f"Total non-null     : {len(kode_raw):,}")
print(f"Length = 16        : {(kode_raw.str.len()==16).sum():,}")
print(f"Length != 16       : {len(wrong_len):,}  <- should be 0")
print(f"Last digit != 0    : {len(last_nonzero):,}")
print(f"Sample values      : {kode_raw.head(3).tolist()}")

if len(last_nonzero):
    print(f"\nSample codes where last digit != 0 (first 5):")
    for code in last_nonzero.unique()[:5]:
        print(f"  Python reads: {code}   <- TRUE stored value")
        print(f"  Excel shows : {code[:-1]}0  <- Excel rounds display to 15 sig. digits")

print("""
WHY THE LAST DIGIT DIFFERS FROM EXCEL DISPLAY:
  Excel stores numbers as float64 (IEEE 754 double precision).
  Float64 only displays 15 significant digits accurately.
  A 16-digit Kode like 3315101201612301:
    -> Python reads  : 3315101201612301  (TRUE stored value)
    -> Excel displays: 3315101201612300  (rounds last digit in display)

  Python is NOT wrong. Excel hides the 16th digit in its display.
  This is a known Excel limitation, not a data error.

ACTION: Do NOT change last digits to 0.
  Kode is only a reference ID -- all joins use bps_kab_code (4-digit).
""")



STEP 3 -- Read & compile annual files
  Files in DATA_DIR (11 xlsx):
    Data Bencana_2016.xlsx
    Data Bencana_2017.xlsx
    Data Bencana_2018.xlsx
    Data Bencana_2019.xlsx
    Data Bencana_2020.xlsx
    Data Bencana_2021.xlsx
    Data Bencana_2022.xlsx
    Data Bencana_2023.xlsx
    Data Bencana_2024.xlsx
    Data Bencana_2025.xlsx
    ~$Data Bencana_2016.xlsx
  [OK] 2016: 825 rows  | Data Bencana_2016.xlsx
  [OK] 2017: 980 rows  | Data Bencana_2017.xlsx
  [OK] 2018: 884 rows  | Data Bencana_2018.xlsx
  [OK] 2019: 790 rows  | Data Bencana_2019.xlsx
  [OK] 2020: 1,609 rows  | Data Bencana_2020.xlsx
  [OK] 2021: 1,931 rows  | Data Bencana_2021.xlsx
  [OK] 2022: 1,573 rows  | Data Bencana_2022.xlsx
  [OK] 2023: 1,265 rows  | Data Bencana_2023.xlsx
  [OK] 2024: 1,510 rows  | Data Bencana_2024.xlsx
  [OK] 2025: 1,702 rows  | Data Bencana_2025.xlsx

Combined : 13,069 rows x 17 cols

STEP 3b -- Kode Identitas Bencana audit
dtype              : object  (must be object/string)
Total non-nu

## STEP 4 — Light cleaning

Column renaming, date parsing, and basic string-field tidying.

In [56]:
# =============================================================================
# STEP 4 -- LIGHT CLEANING
# =============================================================================

print("=" * 60)
print("STEP 4 -- Light cleaning")
print("=" * 60)

combined.rename(columns={
    "ID Kabupaten/Kota": "ID Kabupaten",
    "Kabupaten/Kota"   : "Kabupaten",
}, inplace=True)

if "Tanggal Kejadian" in combined.columns:
    combined["Tanggal Kejadian"] = pd.to_datetime(
        combined["Tanggal Kejadian"], format="%d/%m/%Y", errors="coerce"
    ).fillna(pd.to_datetime(combined["Tanggal Kejadian"], errors="coerce"))

for col in ["ID Kabupaten", "Kabupaten", "Provinsi", "Kejadian"]:
    if col in combined.columns:
        combined[col] = (
            combined[col].astype(str).str.strip().str.upper()
            .replace({"NAN": np.nan, "": np.nan})
        )

print("Column rename, date parse, string normalisation done.")


STEP 4 -- Light cleaning
Column rename, date parse, string normalisation done.


## STEP 5 — Fill blanks

All columns are strings at this point. Blank strings are checked **before** numeric coercion, then filled: numeric fields → 0, text fields → "-".

In [58]:
# =============================================================================
# STEP 5 -- FILL BLANKS
#
# All columns are strings at this point (dtype=str from read_excel).
# We check for blank strings BEFORE any numeric coercion.
#
# Rules:
#   Numeric columns  : blank / NaN / non-numeric string -> 0
#   Text columns     : blank / NaN                      -> "-"
# =============================================================================

print("\n" + "=" * 60)
print("STEP 5 -- Fill blanks  (numeric -> 0,  text -> '-')")
print("=" * 60)

print(f"\n{'Column':<25} {'Blank/NaN':>10}  {'Fill':>6}  {'dtype after':>12}")
print(f"{'─'*25} {'─'*10}  {'─'*6}  {'─'*12}")

for col in NUMERIC_COLS:
    if col not in combined.columns:
        continue
    n_blank = combined[col].apply(is_blank).sum()
    combined[col] = combined[col].apply(
        lambda x: "0" if is_blank(x) else str(x).strip()
    )
    combined[col] = pd.to_numeric(combined[col], errors="coerce").fillna(0)
    print(f"  {col:<23} {n_blank:>10,}  {'-> 0':>6}  {str(combined[col].dtype):>12}")

print()
for col in TEXT_COLS:
    if col not in combined.columns:
        continue
    n_blank = combined[col].apply(is_blank).sum()
    combined[col] = combined[col].apply(
        lambda x: "-" if is_blank(x) else str(x).strip()
    )
    print(f"  {col:<23} {n_blank:>10,}  {chr(39)+'-'+chr(39):>6}  {'object':>12}")

print("\nRemaining NaN after fill (should all be 0):")
chk = [c for c in NUMERIC_COLS + TEXT_COLS if c in combined.columns]
nan_left = combined[chk].isna().sum()
if nan_left.sum() == 0:
    print("  All numeric and text columns fully filled ✓")
else:
    print(nan_left[nan_left > 0].to_string())



STEP 5 -- Fill blanks  (numeric -> 0,  text -> '-')

Column                     Blank/NaN    Fill   dtype after
───────────────────────── ──────────  ──────  ────────────
  Meninggal                    4,219    -> 0         int64
  Hilang                       4,385    -> 0         int64
  Terluka                      4,338    -> 0         int64
  Rumah Rusak                      1    -> 0         int64
  Rumah Terendam               1,252    -> 0         int64
  Fasum Rusak                      0    -> 0         int64

  Lokasi                       2,871     '-'        object
  Penyebab                     6,125     '-'        object
  Kronologi & Dokumentasi          0     '-'        object

Remaining NaN after fill (should all be 0):
  All numeric and text columns fully filled ✓


## STEP 6 — Duplicate inspection and removal

Identify and remove duplicate rows.

In [60]:
# =============================================================================
# STEP 6 -- DUPLICATE INSPECTION & REMOVAL
# =============================================================================

print("\n" + "=" * 60)
print("STEP 6 -- Duplicate inspection & removal")
print("=" * 60)

DUPLICATE_KEYS = [
    "Tanggal Kejadian", "Kejadian", "Kabupaten", "Provinsi",
    "Meninggal", "Hilang", "Terluka",
    "Rumah Rusak", "Rumah Terendam", "Fasum Rusak",
]
active_keys = [k for k in DUPLICATE_KEYS if k in combined.columns]

dup_mask = combined.duplicated(subset=active_keys, keep=False)
dup_df   = combined[dup_mask].copy()

print(f"Duplicate key fields : {active_keys}")
print(f"Total rows involved  : {len(dup_df)}")

freq = (
    dup_df.groupby(active_keys, dropna=False)
    .agg(
        count     = ("year", "size"),
        years     = ("year", lambda x: sorted(x.unique().tolist())),
        kode_list = ("Kode Identitas Bencana", lambda x: x.unique().tolist()),
    )
    .sort_values("count", ascending=False)
    .reset_index()
)
print("\nFrequency breakdown:")
print(freq["count"].value_counts().sort_index()
      .rename_axis("appears N times").to_frame("# groups"))
print("\nTop 10 duplicated groups:")
print(freq.head(10).to_string(index=False))

before = len(combined)
combined = combined.drop(
    index=combined[combined.duplicated(subset=active_keys, keep="first")].index
).reset_index(drop=True)
after = len(combined)

print(f"\nRows before  : {before:,}")
print(f"Rows dropped : {before - after:,}")
print(f"Rows after   : {after:,}")
print(f"Remaining    : {combined.duplicated(subset=active_keys).sum()}  <- should be 0")



STEP 6 -- Duplicate inspection & removal
Duplicate key fields : ['Tanggal Kejadian', 'Kejadian', 'Kabupaten', 'Provinsi', 'Meninggal', 'Hilang', 'Terluka', 'Rumah Rusak', 'Rumah Terendam', 'Fasum Rusak']
Total rows involved  : 77

Frequency breakdown:
                 # groups
appears N times          
2                      37
3                       1

Top 10 duplicated groups:
Tanggal Kejadian Kejadian      Kabupaten         Provinsi  Meninggal  Hilang  Terluka  Rumah Rusak  Rumah Terendam  Fasum Rusak  count  years                                  kode_list
      2021-12-15   BANJIR           POSO  SULAWESI TENGAH          0       0        0            0               0            0      3 [2021]                   [7204 101 2021 012 15 1]
      2017-03-08   BANJIR        BANDUNG       JAWA BARAT          0       0        0            0               0            0      2 [2017]       [3204101201703081, 3204101201703082]
      2024-03-06   BANJIR        JOMBANG       JAWA TIMUR    

## STEP 7 — Name normalisation

Apply the STEP 2 correction dictionaries to standardise regency and province names.

In [62]:
# =============================================================================
# STEP 7 -- NAME NORMALISATION
# =============================================================================

print("\n" + "=" * 60)
print("STEP 7 -- Name normalisation")
print("=" * 60)

combined["bnpb_regency_norm"] = (
    combined["Kabupaten"].str.upper().str.strip()
    .replace(BNPB_REGENCY_CORRECTIONS)
)
combined["bnpb_province_norm"] = (
    combined["Provinsi"].str.upper().str.strip()
    .replace(BNPB_PROVINCE_CORRECTIONS)
)

mask = combined["Kabupaten"].str.upper().str.strip() != combined["bnpb_regency_norm"]
print(f"Regency corrections  : {mask.sum()} rows")
print(combined[mask][["Kabupaten", "bnpb_regency_norm"]]
      .drop_duplicates().to_string(index=False))

mask = combined["Provinsi"].str.upper().str.strip() != combined["bnpb_province_norm"]
print(f"\nProvince corrections : {mask.sum()} rows")
print(combined[mask][["Provinsi", "bnpb_province_norm"]]
      .drop_duplicates().to_string(index=False))



STEP 7 -- Name normalisation
Regency corrections  : 422 rows
                       Kabupaten            bnpb_regency_norm
              LABUHAN BATU UTARA            LABUHANBATU UTARA
            PANGKAJENE KEPULAUAN     PANGKAJENE DAN KEPULAUAN
            KOTA JAKARTA SELATAN ADMINISTRASI JAKARTA SELATAN
              KOTA JAKARTA TIMUR   ADMINISTRASI JAKARTA TIMUR
              KOTA JAKARTA BARAT   ADMINISTRASI JAKARTA BARAT
KEPULAUAN SIAU TAGULANDANG BIARO  KEP. SIAU TAGULANDANG BIARO
              KOTA JAKARTA UTARA   ADMINISTRASI JAKARTA UTARA
              KOTA PANGKALPINANG          KOTA PANGKAL PINANG
                    TOBA SAMOSIR                         TOBA
                  KOTA PARE-PARE                     PAREPARE
              KOTA TANJUNG BALAI            KOTA TANJUNGBALAI
           KOTA PEMATANG SIANTAR         KOTA PEMATANGSIANTAR
                     KOTA BAUBAU                      BAU BAU
                    MAHAKAM HULU                  MAHAKAM ULU
        

## STEP 8 — Build master regency table

Construct the master list of unique regencies/municipalities from the cleaned records.

In [64]:
# =============================================================================
# STEP 8 -- BUILD MASTER REGENCY TABLE
# =============================================================================

print("\n" + "=" * 60)
print("STEP 8 -- Build master regency table")
print("=" * 60)

master_regency = (
    combined[["ID Kabupaten", "bnpb_regency_norm", "bnpb_province_norm"]]
    .dropna(subset=["ID Kabupaten"])
    .rename(columns={
        "bnpb_regency_norm" : "bnpb_regency",
        "bnpb_province_norm": "bnpb_province",
    })
    .assign(
        bnpb_code=lambda d: (
            d["ID Kabupaten"].astype(str)
            .str.extract(r"^(\d{4})", expand=False)
            .str.zfill(4)
        )
    )
    .drop_duplicates(subset=["bnpb_code", "bnpb_regency", "bnpb_province"])
    .sort_values(["bnpb_code", "bnpb_regency"])
    .reset_index(drop=True)
)

print(f"Unique regency entries : {len(master_regency)}")
inconsistent = (
    master_regency.groupby("bnpb_code")
    .filter(lambda g: g["bnpb_regency"].nunique() > 1)
    .sort_values("bnpb_code")
)
if len(inconsistent):
    print(f"Codes with multiple spellings : {inconsistent['bnpb_code'].nunique()}")
    print(inconsistent[["bnpb_code", "bnpb_regency", "bnpb_province"]]
          .to_string(index=False))
else:
    print("No inconsistent spellings ✓")



STEP 8 -- Build master regency table
Unique regency entries : 512
No inconsistent spellings ✓


## STEP 9 — Load BPS master and build lookup tables

Load the BPS↔Kemendagri bridging table and build the lookup tables used for code matching.

*STEP 9b — Papua code remap:* built dynamically from the BPS master to handle the Papua province split.

In [66]:
# ===========================================================================
# STEP 9 -- LOAD BPS MASTER & BUILD LOOKUP TABLES
# =============================================================================

print("\n" + "=" * 60)
print("STEP 9 -- Load BPS master")
print("=" * 60)

bps_raw = pd.read_excel(BPS_PATH)
bps_raw.columns = bps_raw.columns.str.strip().str.lower().str.replace(" ", "_")
if "no" in bps_raw.columns:
    bps_raw = bps_raw.drop(columns=["no"])

bps_df = bps_raw.copy()
bps_df["kemendagri_kab_code_4"]     = bps_df["kemendagri_kab_code"].apply(reformat_kemendagri_code)
bps_df["kemendagri_kab_name_clean"] = bps_df["kemendagri_kab_name"].apply(strip_prefix)
bps_df["bps_kab_name_clean"]        = bps_df["bps_kab_name"].apply(strip_prefix)
bps_df["bps_kab_code"]              = bps_df["bps_kab_code"].astype(str).str.zfill(4)
bps_df["bps_prov_code"]             = bps_df["bps_prov_code"].astype(str).str.zfill(2)

bps_ref = (
    bps_df[["kemendagri_kab_code_4", "kemendagri_kab_name_clean"]]
    .dropna().drop_duplicates()
    .rename(columns={
        "kemendagri_kab_code_4"    : "_code",
        "kemendagri_kab_name_clean": "_name",
    })
)

name_to_kemendagri = (
    bps_df[["kemendagri_kab_name_clean", "kemendagri_kab_code_4"]]
    .dropna()
    .drop_duplicates(subset=["kemendagri_kab_name_clean"])
    .set_index("kemendagri_kab_name_clean")["kemendagri_kab_code_4"]
)

print(f"BPS master records        : {len(bps_df)}")
print(f"bps_ref entries           : {len(bps_ref)}")
print(f"name_to_kemendagri entries: {len(name_to_kemendagri)}")


# =============================================================================
# STEP 9b -- BUILD PAPUA CODE REMAP DYNAMICALLY FROM BPS MASTER
# =============================================================================

print("\n" + "=" * 60)
print("STEP 9b -- Build Papua code remap")
print("=" * 60)

BNPB_CODE_REMAP = {}
for old_code, regency_name in PAPUA_OLD_CODE_TO_NAME.items():
    if old_code in PAPUA_CODE_DIRECT:
        BNPB_CODE_REMAP[old_code] = PAPUA_CODE_DIRECT[old_code]
        continue
    kem_code = name_to_kemendagri.get(strip_prefix(regency_name))
    if kem_code:
        BNPB_CODE_REMAP[old_code] = kem_code
    else:
        print(f"  [WARN] No kemendagri code: {old_code} -> '{regency_name}'")

print(f"BNPB_CODE_REMAP entries : {len(BNPB_CODE_REMAP)}")
missing_targets = (
    set(BNPB_CODE_REMAP.values()) -
    set(bps_df["kemendagri_kab_code_4"].dropna())
)
if missing_targets:
    print(f"[WARN] Targets not in kemendagri: {missing_targets}")
else:
    print("All target codes verified in kemendagri ✓")

print("\nFull remap table:")
for k, v in sorted(BNPB_CODE_REMAP.items()):
    name = PAPUA_OLD_CODE_TO_NAME.get(k, "?")
    print(f"  {k} -> {v}  ({name})")



STEP 9 -- Load BPS master
BPS master records        : 514
bps_ref entries           : 514
name_to_kemendagri entries: 488

STEP 9b -- Build Papua code remap
BNPB_CODE_REMAP entries : 54
All target codes verified in kemendagri ✓

Full remap table:
  9101 -> 9301  (MERAUKE)
  9102 -> 9501  (JAYAWIJAYA)
  9103 -> 9103  (JAYAPURA)
  9104 -> 9401  (NABIRE)
  9105 -> 9105  (KEPULAUAN YAPEN)
  9106 -> 9106  (BIAK NUMFOR)
  9108 -> 9403  (PANIAI)
  9109 -> 9404  (MIMIKA)
  9110 -> 9110  (SARMI)
  9111 -> 9111  (KEEROM)
  9112 -> 9502  (PEGUNUNGAN BINTANG)
  9113 -> 9503  (YAHUKIMO)
  9114 -> 9504  (TOLIKARA)
  9115 -> 9115  (WAROPEN)
  9119 -> 9119  (SUPIORI)
  9120 -> 9120  (MAMBERAMO RAYA)
  9123 -> 9507  (LANNY JAYA)
  9126 -> 9406  (DOGIYAI)
  9128 -> 9408  (DEIYAI)
  9202 -> 9202  (MANOKWARI)
  9203 -> 9203  (FAK FAK)
  9205 -> 9603  (RAJA AMPAT)
  9206 -> 9206  (TELUK BINTUNI)
  9207 -> 9207  (TELUK WONDAMA)
  9208 -> 9208  (KAIMANA)
  9209 -> 9604  (TAMBRAUW)
  9210 -> 9605  (MAYBRAT)


## STEP 10 — Create `bnpb_code_new` (3-tier matching)

Assign BPS codes using a three-tier strategy: (10a) explicit Papua remap, (10b) simultaneous code-and-name match, and (10c) name-only fallback for code mismatches or typos.

In [68]:
# =============================================================================
# STEP 10 -- CREATE bnpb_code_new (3-tier matching)
#
# Tier 10a: Explicit Papua remap  (old code -> correct new kemendagri code)
# Tier 10b: Code AND name simultaneously  (most reliable general match)
# Tier 10c: Name-only fallback  (handles code mismatches / typos)
# =============================================================================

print("\n" + "=" * 60)
print("STEP 10 -- Create bnpb_code_new (3-tier matching)")
print("=" * 60)

mr = master_regency.copy()
mr["bnpb_regency_clean"] = mr["bnpb_regency"].apply(strip_prefix)

# 10a. Explicit Papua remap
mr["bnpb_code_new"] = mr["bnpb_code"].map(BNPB_CODE_REMAP)
mr["match_status"]  = np.where(mr["bnpb_code_new"].notna(), "explicit_remap", np.nan)
n_a = mr["bnpb_code_new"].notna().sum()
print(f"10a. Explicit remap      : {n_a}")

# 10b. Code AND Name simultaneously
no_match = mr["bnpb_code_new"].isna()
step_b = mr[no_match][["bnpb_code", "bnpb_regency_clean"]].merge(
    bps_ref,
    left_on  = ["bnpb_code",         "bnpb_regency_clean"],
    right_on = ["_code",              "_name"],
    how      = "left",
)
matched_b = step_b["_code"].notna()
mr.loc[no_match, "bnpb_code_new"] = np.where(matched_b, step_b["_code"].values, np.nan)
mr.loc[no_match, "match_status"]  = np.where(matched_b, "code_and_name", np.nan)
n_b = (mr["match_status"] == "code_and_name").sum()
print(f"10b. Code AND Name match : {n_b}")

# 10c. Name-only fallback
no_match = mr["bnpb_code_new"].isna()
name_ref = bps_ref.drop_duplicates(subset=["_name"])
step_c = mr[no_match][["bnpb_regency_clean"]].merge(
    name_ref,
    left_on  = "bnpb_regency_clean",
    right_on = "_name",
    how      = "left",
)
matched_c = step_c["_code"].notna()
mr.loc[no_match, "bnpb_code_new"] = np.where(matched_c, step_c["_code"].values, np.nan)
mr.loc[no_match, "match_status"]  = np.where(matched_c, "name_only", np.nan)
n_c = (mr["match_status"] == "name_only").sum()
print(f"10c. Name-only fallback  : {n_c}")

# 10d. Unmatched
mr.loc[mr["bnpb_code_new"].isna(), "match_status"] = "unmatched"
n_d = (mr["match_status"] == "unmatched").sum()
print(f"10d. Unmatched           : {n_d}")

print(f"\nMatch status breakdown:")
print(mr["match_status"].value_counts().to_string())

changed = mr[mr["bnpb_code"] != mr["bnpb_code_new"]].dropna(subset=["bnpb_code_new"])
print(f"\nCodes changed (bnpb_code -> bnpb_code_new): {len(changed)}")
print(changed[["bnpb_code", "bnpb_code_new", "bnpb_regency",
               "bnpb_province", "match_status"]].to_string(index=False))

still_unmatched = mr[mr["match_status"] == "unmatched"]
if len(still_unmatched):
    print(f"\n[WARN] Manual review needed ({len(still_unmatched)}):")
    print(still_unmatched[["bnpb_code", "bnpb_regency", "bnpb_province"]]
          .to_string(index=False))

master_regency = mr.copy()



STEP 10 -- Create bnpb_code_new (3-tier matching)
10a. Explicit remap      : 41
10b. Code AND Name match : 469
10c. Name-only fallback  : 2
10d. Unmatched           : 0

Match status breakdown:
match_status
code_and_name     469
explicit_remap     41
name_only           2

Codes changed (bnpb_code -> bnpb_code_new): 16
bnpb_code bnpb_code_new       bnpb_regency bnpb_province   match_status
     9101          9301            MERAUKE         PAPUA explicit_remap
     9102          9501         JAYAWIJAYA         PAPUA explicit_remap
     9104          9401             NABIRE         PAPUA explicit_remap
     9108          9403             PANIAI         PAPUA explicit_remap
     9109          9404             MIMIKA         PAPUA explicit_remap
     9112          9502 PEGUNUNGAN BINTANG         PAPUA explicit_remap
     9113          9503           YAHUKIMO         PAPUA explicit_remap
     9114          9504           TOLIKARA         PAPUA explicit_remap
     9123          9507       

## STEP 11 — Full outer join → comparison / QC report

Full outer join between BNPB and BPS master to produce a matching quality-control report (matched / unmatched).

In [70]:
# =============================================================================
# STEP 11 -- FULL OUTER JOIN -> COMPARISON / QC REPORT
# =============================================================================

print("\n" + "=" * 60)
print("STEP 11 -- Full outer join -> comparison / QC report")
print("=" * 60)

bps_join = (
    bps_df[[
        "bps_kab_code",          "bps_kab_name",          "bps_kab_name_clean",
        "bps_prov_name",         "bps_prov_code",
        "kemendagri_prov_name",  "kemendagri_prov_code",
        "kemendagri_kab_name",   "kemendagri_kab_code",
        "kemendagri_kab_code_4", "kemendagri_kab_name_clean",
    ]]
    .drop_duplicates(subset=["kemendagri_kab_code_4"])
)

comparison = master_regency.merge(
    bps_join,
    left_on  = "bnpb_code_new",
    right_on = "kemendagri_kab_code_4",
    how      = "outer",
)

comparison["source"] = "both"
comparison.loc[comparison["bnpb_code"].isna(),             "source"] = "BPS/Kemendagri only"
comparison.loc[comparison["kemendagri_kab_code_4"].isna(), "source"] = "BNPB only"

comparison["bnpb_regency_clean"] = comparison["bnpb_regency"].apply(strip_prefix)
comparison["code_matched"]   = (
    comparison["bnpb_code_new"].notna() &
    comparison["kemendagri_kab_code_4"].notna()
)
comparison["kab_name_match"] = (
    comparison["bnpb_regency_clean"] == comparison["kemendagri_kab_name_clean"]
)
comparison["prov_name_match"] = (
    comparison["bnpb_province"].str.upper().str.strip() ==
    comparison["kemendagri_prov_name"].str.upper().str.strip()
)

col_order = [
    "source",               "match_status",
    "bnpb_code",            "bnpb_code_new",
    "bnpb_province",        "bnpb_regency",
    "bps_prov_name",        "bps_prov_code",
    "bps_kab_name",         "bps_kab_code",
    "kemendagri_prov_name", "kemendagri_prov_code",
    "kemendagri_kab_name",  "kemendagri_kab_code",  "kemendagri_kab_code_4",
    "code_matched",         "kab_name_match",        "prov_name_match",
    "bnpb_regency_clean",   "kemendagri_kab_name_clean",
]
col_order = [c for c in col_order if c in comparison.columns]
comparison = (
    comparison[col_order]
    .sort_values(["bnpb_code", "kemendagri_kab_code"], na_position="last")
    .reset_index(drop=True)
)
comparison.index += 1
comparison.index.name = "no"
comparison = comparison.reset_index()

in_both   = (comparison["source"] == "both").sum()
bnpb_only = (comparison["source"] == "BNPB only").sum()
bps_only  = (comparison["source"] == "BPS/Kemendagri only").sum()
kab_ok    = comparison.loc[comparison["source"] == "both", "kab_name_match"].sum()
prov_ok   = comparison.loc[comparison["source"] == "both", "prov_name_match"].sum()

print(f"Total rows (full join)  : {len(comparison)}")
print(f"  Both                  : {in_both}  ({in_both/len(comparison)*100:.1f}%)")
print(f"  BNPB only             : {bnpb_only}")
print(f"  BPS only              : {bps_only}")
print(f"Name match              : {kab_ok}/{in_both}  ({kab_ok/in_both*100:.1f}%)")
print(f"Province match          : {prov_ok}/{in_both}  ({prov_ok/in_both*100:.1f}%)")

name_mismatch = comparison[
    (comparison["source"] == "both") & ~comparison["kab_name_match"]
]
if len(name_mismatch):
    print(f"\nRemaining name mismatches ({len(name_mismatch)}):")
    print(name_mismatch[[
        "bnpb_code", "bnpb_code_new", "match_status",
        "bnpb_regency_clean", "kemendagri_kab_name_clean",
    ]].to_string(index=False))

still_unmatched_comp = comparison[comparison["source"] == "BNPB only"]
if len(still_unmatched_comp):
    print(f"\nStill unmatched in join ({len(still_unmatched_comp)}):")
    print(still_unmatched_comp[[
        "bnpb_code", "bnpb_regency", "bnpb_province"
    ]].to_string(index=False))




STEP 11 -- Full outer join -> comparison / QC report
Total rows (full join)  : 524
  Both                  : 512  (97.7%)
  BNPB only             : 0
  BPS only              : 12
Name match              : 512/512  (100.0%)
Province match          : 496/512  (96.9%)


## STEP 12 — Add BPS code to combined data and export

Attach the resolved BPS code to the compiled records and export the final file.

**Output:** `data/processed/1_bencana_compiled_with_bps.xlsx`

In [72]:
# =============================================================================
# STEP 12 -- ADD BPS CODE TO COMBINED & EXPORT
# =============================================================================

print("\n" + "=" * 60)
print("STEP 12 -- Add BPS code to combined & export")
print("=" * 60)

code_lookup = (
    comparison[comparison["source"] == "both"]
    [[
        "bnpb_code", "bnpb_code_new",
        "bps_kab_code", "bps_kab_name",
        "bps_prov_name", "bps_prov_code",
        "kemendagri_kab_code_4", "kemendagri_kab_name",
        "kemendagri_prov_name", "match_status",
    ]]
    .drop_duplicates(subset=["bnpb_code"])
)

print(f"Lookup entries: {len(code_lookup)}")
dupes = code_lookup[code_lookup.duplicated(subset=["bnpb_code"], keep=False)]
if len(dupes):
    print(f"[WARN] bnpb_codes with multiple BPS mappings ({len(dupes)}):")
    print(dupes[["bnpb_code", "bps_kab_code", "bps_kab_name"]].to_string(index=False))
else:
    print("No duplicate bnpb_code mappings ✓")

combined["bnpb_code"] = (
    combined["ID Kabupaten"].astype(str)
    .str.extract(r"^(\d{4})", expand=False)
    .str.zfill(4)
)

before_cols = combined.columns.tolist()
combined = combined.merge(
    code_lookup[[
        "bnpb_code", "bnpb_code_new",
        "bps_kab_code", "bps_kab_name",
        "bps_prov_code", "bps_prov_name",
        "kemendagri_kab_code_4", "kemendagri_kab_name",
        "kemendagri_prov_name", "match_status",
    ]],
    on="bnpb_code", how="left",
)

BPS_NEW_COLS = [
    "bnpb_code", "bnpb_code_new",
    "bps_kab_code", "bps_kab_name",
    "bps_prov_code", "bps_prov_name",
    "kemendagri_kab_code_4", "kemendagri_kab_name",
    "kemendagri_prov_name", "match_status",
]
orig = [c for c in before_cols if c not in BPS_NEW_COLS]
if "ID Kabupaten" in orig:
    idx = orig.index("ID Kabupaten") + 1
    final_order = orig[:idx] + BPS_NEW_COLS + orig[idx:]
else:
    final_order = orig + BPS_NEW_COLS
combined = combined[[c for c in final_order if c in combined.columns]]

n_matched = combined["bps_kab_code"].notna().sum()
n_missing = combined["bps_kab_code"].isna().sum()
print(f"\nTotal rows       : {len(combined):,}")
print(f"BPS code matched : {n_matched:,}  ({n_matched/len(combined)*100:.1f}%)")
print(f"BPS code missing : {n_missing:,}")

if n_missing:
    miss = (
        combined[combined["bps_kab_code"].isna()]
        [["bnpb_code", "bnpb_regency_norm", "bnpb_province_norm"]]
        .drop_duplicates().sort_values("bnpb_code")
    )
    print(f"\nUnmatched bnpb_codes ({miss['bnpb_code'].nunique()} unique):")
    print(miss.to_string(index=False))

# Export
wb = openpyxl.Workbook()
wb.remove(wb.active)

ws_combined = wb.create_sheet("Combined")
write_excel_sheet(ws_combined, combined)

summary = (
    combined.groupby([
        "bnpb_code", "bnpb_regency_norm", "bnpb_province_norm",
        "bps_kab_code", "bps_kab_name",
        "bps_prov_code", "bps_prov_name", "match_status",
    ])
    .size().reset_index(name="record_count")
    .sort_values("bnpb_code")
)
write_excel_sheet(wb.create_sheet("Code Summary"), summary)
write_excel_sheet(wb.create_sheet("Regency Comparison"), comparison)
write_excel_sheet(
    wb.create_sheet("Matched"),
    comparison[comparison["source"] == "both"].reset_index(drop=True),
)
write_excel_sheet(
    wb.create_sheet("BNPB Only"),
    comparison[comparison["source"] == "BNPB only"].reset_index(drop=True),
)
write_excel_sheet(
    wb.create_sheet("BPS Only"),
    comparison[comparison["source"] == "BPS/Kemendagri only"].reset_index(drop=True),
)
if len(name_mismatch):
    write_excel_sheet(wb.create_sheet("Name Mismatches"),
                      name_mismatch.reset_index(drop=True))
if n_missing:
    write_excel_sheet(wb.create_sheet("No BPS Code"),
                      combined[combined["bps_kab_code"].isna()].reset_index(drop=True))

wb.save(OUTPUT_PATH)
print(f"\nExported -> '{OUTPUT_PATH}'")
print("Sheets : Combined | Code Summary | Regency Comparison |")
print("         Matched | BNPB Only | BPS Only | Name Mismatches")
print("\nDone ✓")


STEP 12 -- Add BPS code to combined & export
Lookup entries: 512
No duplicate bnpb_code mappings ✓

Total rows       : 13,030
BPS code matched : 13,030  (100.0%)
BPS code missing : 0

Exported -> '../data/processed/1_bencana_compiled_with_bps.xlsx'
Sheets : Combined | Code Summary | Regency Comparison |
         Matched | BNPB Only | BPS Only | Name Mismatches

Done ✓


---
### Note — Kode Identitas Bencana (16-digit code)

Python reads the **true** value stored in the Excel cell. Excel only *displays* 15 significant digits, so the last digit appears as `0` in the Excel display even when the stored value ends in `1` (e.g. stored `3315101201612301` displays as `3315101201612300`). This is an Excel display limitation, not a pandas bug. The code is a reference ID only and is never used as a join key.